# Working with JSON in Databricks

This notebook demonstrates how to read, query, and manipulate JSON data in Databricks, including:

* Reading JSON files directly
* Extracting nested fields from JSON
* Working with JSON arrays
* Flattening complex nested structures
* JSON functions (get_json_object, from_json, to_json)
* Struct operations
* Exploding arrays
* Schema inference and evolution

In [0]:
-- Read JSON file directly using read_files()
SELECT * 
FROM read_files(
  'dbfs:/databricks-datasets/samples/people/people.json',
  format => 'json'
);

In [0]:
-- Create a table with nested JSON structure for demonstration
CREATE OR REPLACE TEMP VIEW nested_json AS
SELECT 
  1 as id,
  '{"name": "John", "age": 30, "address": {"city": "New York", "zip": "10001"}, "phones": ["123-456-7890", "098-765-4321"]}' as json_data
UNION ALL
SELECT 
  2 as id,
  '{"name": "Jane", "age": 25, "address": {"city": "San Francisco", "zip": "94102"}, "phones": ["555-123-4567"]}' as json_data
UNION ALL
SELECT 
  3 as id,
  '{"name": "Bob", "age": 35, "address": {"city": "Seattle", "zip": "98101"}, "phones": []}' as json_data;

SELECT * FROM nested_json;

In [0]:
create or replace table json_demo as select * from nested_json;
select * from json_demo;

In [0]:
-- Method 1: Using get_json_object() to extract specific fields from JSON string
SELECT 
  id,
  get_json_object(json_data, '$.name') as name,
  get_json_object(json_data, '$.age') as age,
  get_json_object(json_data, '$.address.city') as city,
  get_json_object(json_data, '$.address.zip') as zip_code,
  get_json_object(json_data, '$.phones[0]') as first_phone
FROM nested_json;

In [0]:
SELECT id, json_data:name as name from json_demo -- using colon syntax

In [0]:
-- Method 2: Using from_json() to parse JSON string into structured columns
-- Define schema and parse JSON
SELECT 
  id,
  from_json(
    json_data,
    'name STRING, age INT, address STRUCT<city:STRING, zip:STRING>, phones ARRAY<STRING>'
  ) as parsed_data
FROM nested_json;

In [0]:
-- Method 3: Access nested struct fields using dot notation (colon syntax)
-- First parse JSON, then access fields
WITH parsed AS (
  SELECT 
    id,
    from_json(
      json_data,
      'name STRING, age INT, address STRUCT<city:STRING, zip:STRING>, phones ARRAY<STRING>'
    ) as data
  FROM nested_json
)
SELECT 
  id,
  data.name,
  data.age,
  data.address.city,
  data.address.zip,
  data.phones
FROM parsed;

In [0]:
-- Explode arrays to create one row per array element
WITH parsed AS (
  SELECT 
    id,
    from_json(
      json_data,
      'name STRING, age INT, address STRUCT<city:STRING, zip:STRING>, phones ARRAY<STRING>'
    ) as data
  FROM nested_json
)
SELECT 
  id,
  data.name,
  explode(data.phones) as phone_number
FROM parsed;

In [0]:
-- Working with array functions
WITH parsed AS (
  SELECT 
    id,
    from_json(
      json_data,
      'name STRING, age INT, address STRUCT<city:STRING, zip:STRING>, phones ARRAY<STRING>'
    ) as data
  FROM nested_json
)
SELECT 
  id,
  data.name,
  data.phones,
  size(data.phones) as phone_count,
  array_contains(data.phones, '123-456-7890') as has_specific_phone,
  get(data.phones, 0) as first_phone, -- safely returns null in case of empty array
  slice(data.phones, 1, 2) as first_two_phones -- slicing index starts at 1
FROM parsed;

In [0]:
-- Convert struct/map back to JSON string using to_json()
WITH parsed AS (
  SELECT 
    id,
    from_json(
      json_data,
      'name STRING, age INT, address STRUCT<city:STRING, zip:STRING>, phones ARRAY<STRING>'
    ) as data
  FROM nested_json
)
SELECT 
  id,
  data.name,
  to_json(data.address) as address_json,
  to_json(data) as full_json
FROM parsed;

In [0]:
-- Infer JSON schema automatically from sample data
SELECT schema_of_json(
  '{"name": "John", "age": 30, "address": {"city": "New York", "zip": "10001"}, "phones": ["123-456-7890"]}'
) as inferred_schema;

In [0]:
select * from nested_json

In [0]:
-- Extract multiple top-level fields at once using json_tuple()
-- More efficient than multiple get_json_object calls
SELECT 
  id,
  jt.*
FROM nested_json
LATERAL VIEW json_tuple(json_data, 'name', 'address') jt as name, address; 

-- LATERAL VIEW  - similar to CTE. Works with table generating functions like explode, json_tuple etc

In [0]:
-- Flatten nested JSON to columnar format (complete flattening)
WITH parsed AS (
  SELECT 
    id,
    from_json(
      json_data,
      'name STRING, age INT, address STRUCT<city:STRING, zip:STRING>, phones ARRAY<STRING>'
    ) as data
  FROM nested_json
)
SELECT 
  id,
  data.name as name,
  data.age as age,
  data.address.city as city,
  data.address.zip as zip,
  data.phones as phones
FROM parsed;

In [0]:
-- Working with real nested JSON from IOT devices
SELECT 
  *,
  _metadata.file_path
FROM read_files(
  'dbfs:/databricks-datasets/iot-stream/data-device/*.json.gz',
  format => 'json'
)
LIMIT 5;

In [0]:
-- Handling malformed or optional JSON fields
CREATE OR REPLACE TEMP VIEW mixed_json AS
SELECT 1 as id, '{"name": "Valid", "age": 30}' as json_data
UNION ALL
SELECT 2 as id, '{"name": "Missing Age"}' as json_data
UNION ALL
SELECT 3 as id, '{invalid json}' as json_data;

-- Use try_* functions to handle errors gracefully
SELECT 
  id,
  get_json_object(json_data, '$.name') as name,
  get_json_object(json_data, '$.age') as age,
  CASE 
    WHEN get_json_object(json_data, '$.name') IS NULL 
    THEN 'Invalid JSON'
    ELSE 'Valid'
  END as json_status
FROM mixed_json;

## JSON Functions Reference

### Parsing & Extraction Functions

| Function | Description | Example |
|----------|-------------|----------|
| **`get_json_object(json, path)`** | Extract field from JSON string using JSONPath | `get_json_object(json_data, '$.name')` |
| **`from_json(json, schema)`** | Parse JSON string to struct/map with schema | `from_json(json_data, 'name STRING, age INT')` |
| **`json_tuple(json, field1, ...)`** | Extract multiple fields efficiently | `json_tuple(json_data, 'name', 'age')` |
| **`to_json(struct)`** | Convert struct/map to JSON string | `to_json(address_struct)` |
| **`schema_of_json(json)`** | Infer schema from JSON sample | `schema_of_json('{"name":"John"}')` |

---

### Array Functions for JSON Arrays

| Function | Description | Example |
|----------|-------------|----------|
| **`explode(array)`** | Create one row per array element | `explode(phones)` |
| **`size(array)`** | Get array length | `size(phones)` |
| **`array_contains(array, value)`** | Check if array contains value | `array_contains(phones, '123-4567')` |
| **`slice(array, start, length)`** | Get array subset | `slice(phones, 1, 2)` |
| **`array[index]`** | Access array element by index (0-based) | `phones[0]` |
| **`flatten(array<array>)`** | Flatten nested arrays | `flatten(nested_array)` |

---

### Struct Operations

| Operation | Description | Example |
|-----------|-------------|----------|
| **Dot notation** | Access struct fields | `address.city` |
| **Colon syntax** | Alternative struct access | `data:address:city` |
| **struct()** | Create struct from columns | `struct(city, zip) as address` |
| **named_struct()** | Create struct with field names | `named_struct('city', city, 'zip', zip)` |

---

### JSONPath Syntax (for get_json_object)

* **`$.field`** — Top-level field
* **`$.nested.field`** — Nested field
* **`$.array[0]`** — Array element by index
* **`$.array[*]`** — All array elements
* **`$[0].field`** — Root array element field

---

### Best Practices

✅ **Use `from_json()` for repeated access** — Parsing once is faster than multiple `get_json_object()` calls  
✅ **Define schemas explicitly** — Better performance than schema inference  
✅ **Use `json_tuple()` for multiple top-level fields** — More efficient than multiple extractions  
✅ **Create Delta tables from parsed JSON** — Better query performance than reading raw JSON repeatedly  
✅ **Use `explode()` in CTEs** — Cleaner code when working with nested arrays  

❌ **Avoid `get_json_object()` in tight loops** — Parse once with `from_json()` instead  
❌ **Don't use `SELECT *` with explode** — Combine with `LATERAL VIEW` or use CTEs

In [0]:
-- Create a sample dataset with a complex nested array for demonstration
CREATE OR REPLACE TEMP VIEW sample_nested_array AS
SELECT 
  1 as id,
  '{"user": {"name": "Alice", "roles": ["admin", "editor"], "contacts": [{"type": "email", "value": "alice@example.com"}, {"type": "phone", "value": "555-1234"}], "preferences": {"notifications": {"email": true, "sms": false}, "languages": ["en", "fr"]}}}' as json_data
UNION ALL
SELECT 
  2 as id,
  '{"user": {"name": "Bob", "roles": ["viewer"], "contacts": [{"type": "email", "value": "bob@example.com"}], "preferences": {"notifications": {"email": false, "sms": true}, "languages": ["en"]}}}' as json_data
UNION ALL
SELECT 
  3 as id,
  '{"user": {"name": "Charlie", "roles": [], "contacts": [], "preferences": {"notifications": {"email": false, "sms": false}, "languages": []}}}' as json_data
UNION ALL
SELECT
  4 as id,
  '{"user": {"name": "Dana", "roles": ["editor", "contributor"], "contacts": [{"type": "email", "value": "dana@example.com"}, {"type": "phone", "value": "555-6789"}, {"type": "fax", "value": "555-0000"}], "preferences": {"notifications": {"email": true, "sms": true}, "languages": ["es", "en"]}}}' as json_data;

SELECT * FROM sample_nested_array;

In [0]:
select schema_of_json('{"user": {"name": "Dana", "roles": ["editor", "contributor"], "contacts": [{"type": "email", "value": "dana@example.com"}, {"type": "phone", "value": "555-6789"}, {"type": "fax", "value": "555-0000"}], "preferences": {"notifications": {"email": true, "sms": true}, "languages": ["es", "en"]}}}')

In [0]:
create or replace table nested_json_complex as
select id, from_json(json_data, 'STRUCT<user: STRUCT<contacts: ARRAY<STRUCT<type: STRING, value: STRING>>, name: STRING, preferences: STRUCT<languages: ARRAY<STRING>, notifications: STRUCT<email: BOOLEAN, sms: BOOLEAN>>, roles: ARRAY<STRING>>>') as data from sample_nested_array

In [0]:
select * from nested_json_complex

In [0]:
-- Create a sample dataset with nested arrays for flatten() demonstration
CREATE OR REPLACE TEMP VIEW nested_array_demo AS
SELECT 
  1 as id,
  array(
    array('a', 'b'),
    array('c'),
    array('d', 'e', 'f')
  ) as nested_arr
UNION ALL
SELECT 
  2 as id,
  array(
    array('x', 'y'),
    array('z')
  ) as nested_arr;

-- Use flatten() to convert nested arrays to a single array
SELECT
  id,
  nested_arr,
  flatten(nested_arr) as flat_arr
FROM nested_array_demo;

In [0]:
-- Business Use Case Example: Flattening Customer Purchases Across Multiple Orders

-- Sample dataset: Each customer has multiple orders, each order contains an array of purchased items
CREATE OR REPLACE TEMP VIEW customer_orders AS
SELECT 
  1 as customer_id,
  array(
    array('Laptop', 'Mouse'),
    array('Keyboard'),
    array('Monitor', 'HDMI Cable', 'Mousepad')
  ) as orders
UNION ALL
SELECT 
  2 as customer_id,
  array(
    array('Tablet', 'Stylus'),
    array('Charger')
  ) as orders;

-- Use flatten() to get a single array of all items purchased by each customer
SELECT
  customer_id,
  orders,
  flatten(orders) as all_items_purchased
FROM customer_orders;